# 训练显存三巨头：激活 / 梯度 / 优化器状态

> mokiomind 项目配套学习笔记（Notebook 版）

一次训练，显存都花在哪？很多人只知道"模型参数"，但真正吃掉显存的，是另外三块。本 notebook 用一个仿 mokiomind 的 mini attention block，一步步实测这三块到底各占多少，并回答上一课留下的问题：**为什么 GQA 训练时省显存省得少，而推理时却能省 3/4**。

每个代码格子都写了逐行注释。建议：先跑一遍，再逐行对着注释读一遍，最后把 `display(...)` 前面的 `#` 去掉自己观察。

## 0. 先给出答案，再拆解

训练一次 forward+backward，显存主要流向四块：

| 名称 | 是什么 | 量级（相对参数） |
|---|---|---|
| 参数 Parameters | 模型待训练的权重 | 1 份 |
| 激活值 Activations | 前向保存的中间结果，反向要用 | 每 token 每层都有，**巨大** |
| 梯度 Gradients | 每个参数算出的导数 | 1 份 |
| 优化器状态 Optimizer State | AdamW 每参数多存 2 份 | 2 份 |

下面用代码实测（本机 torch 2.7.1 + CUDA）。

In [10]:
# ---- 导入库：本格只做导入和超参数定义 ----
import torch                    # PyTorch 主库：张量 + 自动求导
import torch.nn as nn           # 网络模块：nn.Linear 等层
import torch.nn.functional as F # 函数式接口：F.silu 激活、F.softmax

torch.manual_seed(0)            # 固定随机种子，保证每次运行数字一致、可复现

# 三个超参数（量级接近 mokiomind）
B      = 2     # batch size：一次喂 2 条样本
T      = 256   # 序列长度：每条样本 256 个 token
hidden = 512   # 隐层维度：每个 token 的向量长度

print('一个 batch 的 token 总数 B*T =', B * T)  # 2×256 = 512 个 token

一个 batch 的 token 总数 B*T = 512


## 1. 造一个仿 mokiomind 的 mini block

由**注意力**（wq/wk/wv/wo）+ **SwiGLU 前馈**（gate/up/down）组成，结构对应 mokiomind 的一个 transformer layer。为聚焦"显存"这个主题，注意力做了最大简化（没写 GQA 分组、没写 mask），这不影响我们要测的显存结论。

In [16]:
class MiniBlock(nn.Module):
    """一个简化版 transformer layer：注意力 + SwiGLU 前馈。"""
    def __init__(self, hidden):
        super().__init__()
        # ---- 注意力部分的 4 个投影矩阵 ----
        self.wq   = nn.Linear(hidden, hidden)          # 输入 x → Query
        self.wk   = nn.Linear(hidden, hidden)          # 输入 x → Key
        self.wv   = nn.Linear(hidden, hidden)          # 输入 x → Value
        self.wo   = nn.Linear(hidden, hidden)          # 注意力输出 → 回去(输出投影)
        # ---- SwiGLU 前馈的 3 个矩阵 ----
        self.gate = nn.Linear(hidden, 4 * hidden)      # 门控：SiLU(gate(x))
        self.up   = nn.Linear(hidden, 4 * hidden)      # 内容：up(x)
        self.down = nn.Linear(4 * hidden, hidden)      # 4×hidden → 收回到 hidden

    def forward(self, x):
        # x 形状: (B, T, hidden)

        # ---- 注意力：先把 x 投影出 Q、K、V ----
        q = self.wq(x)                      # (B,T,hidden) Query，用于"我关心什么"
        k = self.wk(x)                      # (B,T,hidden) Key，用于"我能匹配什么"
        v = self.wv(x)                      # (B,T,hidden) Value，用于"我携带什么信息"

        # 点积注意力：Q 与 K 转置做矩阵乘，再除以 sqrt(hidden) 防数值爆炸
        attn = q @ k.transpose(-2, -1) / (hidden ** 0.5)  # (B, T, T)

        # softmax 归一化成权重，再加权 Value → 取回上下文信息
        ctx = torch.softmax(attn, dim=-1) @ v            # (B,T,hidden)

        # 输出投影，把注意力结果映射回隐藏空间，完成 attention 部分
        out_attn = self.wo(ctx)                          # (B,T,hidden)

        # ---- SwiGLU 前馈：SiLU(gate)*up，再 down 收尾 ----
        h = F.silu(self.gate(x)) * self.up(x)  # (B,T,4*hidden) 门控+内容逐元素相乘
        return out_attn + self.down(h)         # 残差连接：注意力输出 + 前馈输出

# ---- 实例化模型，并统计参数总量 ----
block = MiniBlock(hidden)                     # 建一个 mini block
x = torch.randn(B, T, hidden)                 # 造一个随机输入，形状和上面注释一致

# 把每个参数的元素个数(sum)累加起来，就是参数总数
n_params = sum(p.numel() for p in block.parameters())

# numel()=元素个数；fp32 每个数占 4 字节，所以 数×4 就是字节，再除 1e6 变兆字节 MB
print(f'参数数 = {n_params:,}   ({n_params*4/1e6:.2f} MB @ fp32)')
print('param.dtype =', next(block.parameters()).dtype)  # 展示默认是 fp32

参数数 = 4,200,960   (16.80 MB @ fp32)
param.dtype = torch.float32


## 2. 第一巨头：激活值 Activations

**为什么一定要存？**

反向传播靠链式法则：
$$
 \frac{\partial L}{\partial w} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial w} = \frac{\partial L}{\partial y} \cdot x
$$
要算梯度的每一项，都需要前向时的**中间结果**（比如上式的 `x`）。所以 forward 出的每一层输出、每个 attention 矩阵，都会被 `autograd` 保存在内存里，等 backward 来用。这批东西就是"激活值"——它是训练显存的**真正大头**。

下面用一个 **forward hook（钩子）** 去"偷看" forward 过程中每个中间张量有多大。hook 是 PyTorch 的机制：在某个层被调用时，自动触发我们挂的 `f` 函数，把该层输出的大小记下来。


**激活规模估算（一个 batch、一层）**：
$$
A_{\text{1 layer}} \;\approx\; \underbrace{B\cdot T\cdot d_{\text{hidden}}}_{\text{每层每个中间张量}} \times K_{\text{中间张量个数}} \;+\; \underbrace{B\cdot T^{\,2}\cdot 1}_{\text{QK}^\top\text{注意力矩阵}}
$$
$$
A_{\text{total}} \;=\; L_{\text{layers}} \cdot A_{\text{1 layer}}
$$
关键结论：激活随 $B\times T$ 线性增长，而注意力矩阵 $\text{QK}^\top$ 随 $T^2$ 增长——这就是长序列训练吃掉海量显存的数学原因。



In [18]:
activation_sizes = {}   # 空字典，用来记录 name -> 元素个数

def make_hook(name):
    """返回一个钩子函数：把某层输出的大小记进 activation_sizes[name]"""
    def f(mod, inp, out):
        # out 可能是一个张量，也可能是一串张量(list/tuple)；统一取元素个数
        activation_sizes[name] = sum(v.numel() for v in (out if isinstance(out, (list, tuple)) else [out]))
    return f

# 给 7 个线性层各挂一个钩子，分别命名：
for nm, m in [('wq→q',  block.wq),   ('wk→k', block.wk),   ('wv→v', block.wv),
              ('wo→out', block.wo),  ('gate', block.gate), ('up', block.up),
              ('down',  block.down)]:
    m.register_forward_hook(make_hook(nm))   # 注册钩子，forward 时自动触发

# 用 no_grad 跑一次（不需要梯度，纯粹为了抓中间张量的大小）
with torch.no_grad():
    block(x)

# 把记录的中间张量元素数求和 = 单层激活总量
n_act = sum(activation_sizes.values())
print(f'单层输出的激活总和 = {n_act:,}  ({n_act*4/1e6:.2f} MB @ fp32)')
print(f'QK^T 注意力矩阵本身 = B*T*T = {B*T*T:,}')   # 注意它随 T² 增长
print(f'放大到 L 层 × 一次 batch，就是 L × 上面的量级')
print(activation_sizes)   # 打印每一块的明细

单层输出的激活总和 = 3,407,872  (13.63 MB @ fp32)
QK^T 注意力矩阵本身 = B*T*T = 131,072
放大到 L 层 × 一次 batch，就是 L × 上面的量级
{'wq→q': 262144, 'wk→k': 262144, 'wv→v': 262144, 'wo→out': 262144, 'gate': 1048576, 'up': 1048576, 'down': 262144}


**关键认知**：激活的大小正比于 `B × T × (hidden 维度)`，跟着**序列长度和 batch**一起涨。这解释了为什么大上下文、长序列训练极耗显存——激活随 `T` 线性涨，而 `QK^T` 甚至随 `T²` 涨。

## 3. 第二巨头：梯度 Gradients

每个参数都有一份梯度，大小**恰好等于参数数**（每个权重算一个导数）。`loss.backward()` 就是反向传播，把梯度填进每个 `p.grad`。


**每个参数的梯度（链式法则）**：
$$
g_W \;=\; \frac{\partial L}{\partial W} \;=\; \frac{\partial L}{\partial y}\cdot \frac{\partial y}{\partial W} \;=\; \frac{\partial L}{\partial y}\cdot x
$$
梯度张量形状与对应参数相同，元素数等于参数元素数：
$$
N_{\text{grad}} \;=\; N_{\text{param}}
$$



In [13]:
# ---- 跑一次完整前向 + 反向，把梯度算出来 ----
out = block(x)                       # 前向：输入 x 过一遍网络
loss = out.pow(2).mean()             # 造一个 loss：输出平方的均值（简单目标）
loss.backward()                      # 反向传播：自动计算每个参数的梯度 p.grad

# 统计所有参数的梯度元素数。只有参与求导的参数才有 grad（用 if 判空保险）
n_grad = sum(p.grad.numel() for p in block.parameters() if p.grad is not None)
print(f'梯度数 = {n_grad:,}  (== 参数数 {n_params:,})，即每参数 1 份')
print(f'示例：block.wq.weight.grad 形状 = {tuple(block.wq.weight.grad.shape)}')
# 上面这行想表达：一个 (512,512) 的权重，就有一份同样形状 (512,512) 的梯度

梯度数 = 4,200,960  (== 参数数 4,200,960)，即每参数 1 份
示例：block.wq.weight.grad 形状 = (512, 512)


## 4. 第三巨头：优化器状态 Optimizer State

`AdamW` 每个参数除了自身和梯度，还要多维护**两个动量**：

- `exp_avg`（一阶矩 m，动量，类似"速度"）
- `exp_avg_sq`（二阶矩 v，自适应步长，类似"方差"）

所以 AdamW 对一个参数要存 **1(参数)+1(梯度)+2(动量) = 4 份**。优化器状态是参数的 **2 倍**，这是"参数×2"之外的又一次翻倍。


**AdamW 参数更新公式**（第 $t$ 步，梯度为 $g_t$）：
$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t \qquad(\text{一阶矩，动量})
$$
$$
v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^{\,2} \qquad(\text{二阶矩，自适应步长})
$$
$$
\hat m_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^t} \qquad(\text{偏差修正})
$$
$$
\theta_{t+1} = \theta_t - \eta\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon} - \eta\,\lambda\,\theta_t
$$
其中 $m_t$、$v_t$ 就是每个参数额外维护的 2 份状态，因此优化器状态数是参数的 2 倍：
$$
N_{\text{opt}} = 2\,N_{\text{param}} \;\;\Longrightarrow\;\; \theta+g+m+v = 4\,N_{\text{param}}
$$



In [14]:
# ---- 建一个 AdamW 优化器，看它给每个参数多存了 2 份状态 ----
opt = torch.optim.AdamW(block.parameters())   # 优化器接管所有参数
opt.step()   # 必须真正更新一次，状态(exp_avg/exp_avg_sq)才会被创建出来

# opt.state 是按参数分组的状态字典；每个状态里有 exp_avg 和 exp_avg_sq 两个张量
# 把它们(所有 tensor)的元素个数全部累加 = 优化器状态总数
n_opt = sum(t.numel() for g in opt.state.values() for t in g.values() if isinstance(t, torch.Tensor))

print(f'优化器状态数 = {n_opt:,}   ≈ 2 × 参数数({n_params:,})')
print('即：AdamW 里每个参数额外多存 exp_avg 和 exp_avg_sq 两个张量')

优化器状态数 = 8,401,934   ≈ 2 × 参数数(4,200,960)
即：AdamW 里每个参数额外多存 exp_avg 和 exp_avg_sq 两个张量


In [15]:
# ---- 把三巨头汇总成一张表 ----
print('=== 三巨头汇总（单层、单 batch、fp32） ===')
#  :>12 是右对齐占位，好看对齐；*4/1e6 = 元素数×4字节→MB
print(f'参数       : {n_params:>12,}   {n_params*4/1e6:6.2f} MB')
print(f'梯度       : {n_grad:>12,}   {n_grad*4/1e6:6.2f} MB')
print(f'优化器状态 : {n_opt:>12,}   {n_opt*4/1e6:6.2f} MB')
print(f'单层激活   : {n_act:>12,}   {n_act*4/1e6:6.2f} MB  (×层数×就是全网络)')
print()
print('结论：想省显存，优先打 激活值 和 优化器状态 的主意，而不是参数。')

=== 三巨头汇总（单层、单 batch、fp32） ===
参数       :    4,200,960    16.80 MB
梯度       :    4,200,960    16.80 MB
优化器状态 :    8,401,934    33.61 MB
单层激活   :    3,407,872    13.63 MB  (×层数×就是全网络)

结论：想省显存，优先打 激活值 和 优化器状态 的主意，而不是参数。


## 5. 回到上一课：为什么 GQA 训练省得少？

推理时 GQA 靠"**不缓存历史 K/V**"省下 KV Cache 的 3/4。但 KV Cache **只在推理存在**。

训练时根本没有 KV Cache，显存大头是上面三巨头——而 GQA 对它们几乎不友好：

- **优化器状态 / 梯度**：每个参数都有一条，GQA 只是把 K/V 拆成更少的头，**参数总量几乎不变**，所以这两块 GQA 省不动。
- **激活值**：训练里 `repeat_kv` 是**真的展开**的——那"复制成 8 份"会在计算图里物化成 8 份参与 forward + backward，所以 attention 部分的激活 GQA 也省不掉。

一句话：**GQA 的显存红利几乎全部来自推理的 KV Cache；训练上它只能省掉一点点 K/V 投影的激活，在所有者面前微不足道。**

> 这也解释了为什么业界常说 GQA 主要是"**推理优化**"手段，而不是"训练省显存"手段。

## 6. 一句话小结

- **激活值**：前向存的中间结果，反向要用 → 训练显存最大头，随 B×T 涨。
- **梯度**：每参数 1 份。
- **优化器状态**：AdamW 每参数再 2 份（m、v），合计是参数的 4 倍内存。
- **GQA**：只有推理省 KV Cache；训练三巨头它省不动。

下一课建议：既然激活这么大，可以接着看 **梯度检查点（gradient checkpointing）**、**混合精度（bf16）** 怎么把这三巨头压下去——这些都是你 mokiomind 训练脚本里会用的技术。


**一次训练的峰值显存估算（fp32，N 参数、L 层、一个 batch）**：
$$
\text{Mem} \;\approx\; \underbrace{4N}_{\text{参数}} + \underbrace{4N}_{\text{梯度}} + \underbrace{8N}_{\text{优化器状态}} + \underbrace{4\,L\,A_{\text{1 layer}}}_{\text{激活值}} \quad \text{(Bytes)}
$$

